In [22]:
#Dataset
#ecommerce_orders_raw.csv

#Objective
#You are a data analyst for an e-commerce company. The company wants to understand its customers,
#products, sales performance, and revenue trends. Use Pandas to transform the raw order data into
#meaningful business information.
#Part A — Data Understanding
import pandas as pd
#1. Load the dataset.

dataset = pd.read_csv("ecommerce_orders_raw.csv")
df = pd.DataFrame(dataset)

#2. Display its structure and dimensions.
print("Shape (rows, columns):", df.shape)
print("Number of dimensions:", df.ndim)

#3. Identify the data types of all columns.
print("\nDataFrame Columns:\n")
df.info()


#4. Identify missing values.
print('Missing values \n', df.isnull())

#5. Identify duplicate orders.
print('Duplicate Orders: \n')
print(df[df["Order_ID"].duplicated(keep=False)], "\n")

#6. Examine the unique values in:
#o City
print('Unique Cities :' , df['City'].unique())
#o Product Category
print('Unique Product categories: ', df['Product_Category'].unique())
#o Payment Method
print('Unique Payment methods: ', df['Payment_Method'].unique())

Shape (rows, columns): (15100, 11)
Number of dimensions: 2

DataFrame Columns:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15100 entries, 0 to 15099
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Order_ID          15100 non-null  object 
 1   Customer_ID       15100 non-null  object 
 2   City              15100 non-null  object 
 3   Product_Category  15100 non-null  object 
 4   Product           15100 non-null  object 
 5   Quantity          15100 non-null  int64  
 6   Unit_Price        15100 non-null  float64
 7   Discount          15100 non-null  int64  
 8   Payment_Method    15100 non-null  object 
 9   Order_Date        15100 non-null  object 
 10  Rating            14699 non-null  float64
dtypes: float64(2), int64(2), object(7)
memory usage: 1.3+ MB
Missing values 
        Order_ID  Customer_ID   City  Product_Category  Product  Quantity  \
0         False        False  False          

In [35]:
#Part B — Data Cleaning
#7. Handle missing ratings.
empty_rows_before = df['Rating'].isna().sum()
df['Rating'] = df['Rating'].fillna(0)
empty_rows_after = df['Rating'].isna().sum()

print('empty_rows_before ::', empty_rows_before)
print('empty_rows_after ::', empty_rows_after)

#8. Standardize city names.
df['City'] = df['City'].astype(str).str.strip().str.title()
print(df['City'].unique())

#9. Identify invalid quantities.
invalid_quantities = df[df['Quantity'] < 0]
print(invalid_quantities)

#10. Handle duplicate orders.
df_no_duplicates = df.drop_duplicates(subset=['Order_ID'], keep='first')
print("\nAfter removing duplicates (keep first):")
print(df_no_duplicates)

#11. Convert Order_Date into an appropriate datetime format.
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce', dayfirst=False)
df['Converted_order_date'] = df['Order_Date'].dt.strftime('%B %d, %Y')
print(df['Converted_order_date'])

empty_rows_before :: 0
empty_rows_after :: 0
['Delhi' 'Pune' 'Hyderabad' 'Mumbai' 'Kolkata' 'Bangalore' 'Chennai'
 'Coimbatore']
      Order_ID Customer_ID       City Product_Category         Product  \
101    O012623      C02458      Delhi  Home Appliances  Vacuum Cleaner   
476    O004601      C02136    Chennai         Clothing           Jeans   
1639   O006694      C01856       Pune           Sports        Football   
2420   O007460      C02632  Bangalore      Electronics     Smart Watch   
4496   O002305      C01604    Kolkata           Sports     Cricket Bat   
4861   O012185      C00664    Chennai            Books           Novel   
5008   O001742      C02508       Pune            Books       Self Help   
6591   O011832      C02379  Hyderabad      Electronics      Smartphone   
8972   O002382      C00456    Kolkata         Clothing           Shoes   
9230   O008580      C01669    Kolkata         Clothing          Jacket   
10292  O014631      C00935    Chennai      Electronics   

In [44]:
#Part C — Feature Engineering
#12. Create:
#Gross_Amount = Quantity × Unit_Price
df['Gross_Amount'] = df['Quantity'] * df['Unit_Price']

#13. Create:
#Discount_Amount = Gross_Amount × Discount / 100
df['Discount_Amount'] = df['Gross_Amount'] * df['Discount'] /100

#14. Create:
#Net_Amount = Gross_Amount - Discount_Amount
df['Net_Amount'] = df['Gross_Amount'] - df['Discount_Amount']

#15. Create a Rating_Category column:
#1–2 → Poor,3 → Average,4 → Good,5 → Excellent
bins = [-1, 2, 3, 4, 5]
labels = ['Poor', 'Average', 'Good', 'Excellent']
df['Rating_category'] = pd.cut(df['Rating'], bins=bins, labels=labels)
df.to_csv('ecommerce_orders_raw_updated.csv', index=False)

#16. Extract the following from Order_Date:
#• Year
print(df['Order_Date'].dt.year)
#• Month
print(df['Order_Date'].dt.month)
#• Day
print(df['Order_Date'].dt.day)
#• Day of Week
#print(df['Order_Date'].dt.week)

0        2026
1        2026
2        2026
3        2025
4        2025
         ... 
15095    2025
15096    2026
15097    2025
15098    2025
15099    2025
Name: Order_Date, Length: 15100, dtype: int32
0         7
1         6
2         7
3         2
4         1
         ..
15095     7
15096     5
15097     7
15098     2
15099    10
Name: Order_Date, Length: 15100, dtype: int32
0        25
1        27
2        25
3         3
4        16
         ..
15095    19
15096    31
15097    26
15098     3
15099     7
Name: Order_Date, Length: 15100, dtype: int32


In [73]:
#Part D — Business Analysis
#17. Calculate total revenue.
df['Revenue'] = df['Unit_Price'] * df['Quantity']
total_revenue = df['Revenue'].sum()
print('Total Revenue: ', total_revenue)

#18. Calculate total number of orders.
print('Total number of orders: ', df['Order_ID'].nunique())

#19. Find the average order value.
print('Average order value: ', df['Net_Amount'].mean().round(2))

#20. Find total revenue by product category.
revenue_by_product = df.groupby('Product_Category')['Revenue'].sum()
print('revenue_by_product :', revenue_by_product)

#21. Find the top 10 products by revenue.
top_10_product_by_revenue = revenue_by_product.head(10)
print('top_10_product_by_revenue:', top_10_product_by_revenue)

#22. Find the city generating the highest revenue.
city_with_higest_revenue = df.groupby('City')['Revenue'].sum().nlargest(1)
print('city_with_higest_revenue :', city_with_higest_revenue)

#23. Find the most frequently used payment method.
most_used_payment_method = df['Payment_Method'].mode()
print('most_used_payment_method: ', most_used_payment_method)

#24. Find the average rating for each product category.
print('average_rating:', df.groupby('Product_Category')['Rating'].mean())

#25. Calculate monthly revenue.
print('Monthly revenue:', df.groupby(df['Order_Date'].dt.to_period('M'))['Revenue'].sum())

#26. Identify the highest-revenue month.
print('highest-revenue month: ', df.groupby(df['Order_Date'].dt.to_period('M'))['Revenue'].sum().idxmax())

#27. Find the top 10 customers by total spending.
print('top 10 customers by total spending:', df.groupby('Customer_ID')['Net_Amount'].sum().nlargest(10))

#28. Compare revenue generated by different cities.
print(' revenue generated by different cities:',df.groupby('City')['Revenue'].sum().sort_values(ascending=False))

#29. Find which product category has the highest average order value.
highest_avg_value = df.groupby('Product_Category')['Unit_Price'].mean().idxmax()
#print(df.groupby('Product_Category')['Unit_Price'].sum())
print('product category with the highest average order value:', highest_avg_value)

Total Revenue:  2276609134.67
Total number of orders:  15000
Average order value:  132006.96
revenue_by_product : Product_Category
Beauty             3.809582e+08
Books              3.826008e+08
Clothing           3.971966e+08
Electronics        3.711161e+08
Home Appliances    3.669061e+08
Sports             3.778313e+08
Name: Revenue, dtype: float64
top_10_product_by_revenue: Product_Category
Beauty             3.809582e+08
Books              3.826008e+08
Clothing           3.971966e+08
Electronics        3.711161e+08
Home Appliances    3.669061e+08
Sports             3.778313e+08
Name: Revenue, dtype: float64
city_with_higest_revenue : City
Chennai    3.260928e+08
Name: Revenue, dtype: float64
most_used_payment_method:  0    Debit Card
Name: Payment_Method, dtype: object
average_rating: Product_Category
Beauty             3.783379
Books              3.759843
Clothing           3.788574
Electronics        3.708266
Home Appliances    3.770665
Sports             3.782154
Name: Rating, d